# Haematopoietic reference integration and correlation

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
# Step 0: resolve repository-relative paths.
project_root <- normalizePath(Sys.getenv("BMO_PROJECT_ROOT", unset = "."), winslash = "/", mustWork = TRUE)
dir.create(file.path(project_root, "data", "processed"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "tables"), recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(SeuratObject)
library(tidyverse)
library(Matrix)
library(pheatmap)


<b><font size=5 color=pink >Step 1: load BMO haematopoietic cells</font></b>


In [ ]:
### ============================================
### ============================================
Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "Haematopoietic.rds"))

Haematopoietic@meta.data$group <- ""
Haematopoietic@meta.data$group[Haematopoietic@meta.data$orig.ident %in% c("Dynamic.25d.1", "Dynamic.25d.2", "Dynamic.25d.3")] <- "Dynamic.25d"
Haematopoietic@meta.data$group[Haematopoietic@meta.data$orig.ident %in% c("Dynamic.31d")] <- "Dynamic.31d"
Haematopoietic@meta.data$group[Haematopoietic@meta.data$orig.ident %in% c("Static.25d.1", "Static.25d.2", "Static.25d.3")] <- "Static.25d"
table(Haematopoietic@meta.data$group)

Haematopoietic@meta.data$celltype <- ""
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("17")] <- "HSC"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("0","1","4","5","6","7","10","12","13")] <- "Erythroid"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("14")] <- "Mast"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("19")] <- "DC"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("3", "9")] <- "Macrophage"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("2", "11", "8")] <- "Monocyte"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("15")] <- "Basophil"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("16")] <- "Eosinophil"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("18")] <- "Neutrophil"

mk_barcodes <- WhichCells(Haematopoietic, expression = PPBP > 1 | PF4 > 1)
valid_mk <- intersect(mk_barcodes, colnames(Haematopoietic))
Haematopoietic@meta.data[valid_mk, "celltype"] <- "Megakaryocyte"
table(Haematopoietic$celltype)

custom_cols_hema <- c(
  "HSC" = "#555555", "Megakaryocyte" = "#FFD92F", "Erythroid" = "#E41A1C",
  "Neutrophil" = "#1F78B4", "Eosinophil" = "#A6CEE3", "Basophil" = "#33A02C",
  "Mast" = "#B2DF8A", "Monocyte" = "#FDBF6F", "Macrophage" = "#B15928", "DC" = "#CAB2D6"
)
Haematopoietic$celltype <- factor(Haematopoietic$celltype, levels = names(custom_cols_hema))

DimPlot(Haematopoietic, reduction = "tsne", group.by = "celltype",
        label = TRUE, cols = custom_cols_hema, pt.size = 0.8) +
  theme(aspect.ratio = 1)


<b><font size=5 color=pink >Step 3: load adult bone marrow</font></b>


In [ ]:
### ============================================
### ============================================
load(file.path(project_root, "data", "reference", "Adult_Bone_Marrow.RData"))

adult_stromal_celltypes <- c("Adipo-MSC","AEC","APOD+ MSC","Fibro-MSC","Osteo-MSC",
                             "Osteoblast","RNAlo MSC","SEC","THY1+ MSC","VSMC")

adult_haem_celltypes <- c("Ba/Eo/Ma","CD4+ T-Cell","CD8+ T-Cell","CLP","Cycling DCs",
                          "Cycling HSPC","Early Myeloid Progenitor","Erythroblast","GMP",
                          "HSC","Late Erythroid","Late Myeloid","Macrophages","Mature B",
                          "Megakaryocyte","MEP","Monocyte","MPP","Neutrophil","pDC",
                          "Plasma Cell","Pre-B","Pre-Pro B","Pro-B","RBC")

#Adult_BM_Stromal <- subset(Adult_Bone_Marrow, subset = cluster_anno_l2 %in% adult_stromal_celltypes)
Adult_BM_Haematopoietic <- subset(Adult_Bone_Marrow, subset = cluster_anno_l2 %in% adult_haem_celltypes)

#Adult_BM_Stromal$cluster_anno_l2 <- droplevels(Adult_BM_Stromal$cluster_anno_l2)
Adult_BM_Haematopoietic$cluster_anno_l2 <- droplevels(Adult_BM_Haematopoietic$cluster_anno_l2)

#table(Adult_BM_Stromal$cluster_anno_l2)
table(Adult_BM_Haematopoietic$cluster_anno_l2)


In [ ]:
# Adult BM stromal
#Adult_BM_Stromal$celltype <- Adult_BM_Stromal$cluster_anno_l2

# Adult BM haematopoietic
Adult_BM_Haematopoietic$celltype <- Adult_BM_Haematopoietic$cluster_anno_l2

#table(Adult_BM_Stromal$celltype)
table(Adult_BM_Haematopoietic$celltype)


<b><font size=5 color=pink >Step 4: load fetal bone marrow</font></b>


In [ ]:
### ============================================
### ============================================
load(file.path(project_root, "data", "reference", "FBM_obj_2.RData"))

FBM_obj_2$compartment <- ifelse(
  FBM_obj_2$broad_fig1_cell.labels == "stroma", "Stromal", "Haematopoietic"
)

#FBM_Stromal <- subset(FBM_obj_2, subset = compartment == "Stromal")
FBM_Haematopoietic <- subset(FBM_obj_2, subset = compartment == "Haematopoietic")

#FBM_Stromal$broad_fig1_cell.labels <- droplevels(FBM_Stromal$broad_fig1_cell.labels)
FBM_Haematopoietic$broad_fig1_cell.labels <- droplevels(FBM_Haematopoietic$broad_fig1_cell.labels)
#FBM_Stromal$cell.labels <- droplevels(FBM_Stromal$cell.labels)

#table(FBM_Stromal$cell.labels)
table(FBM_Haematopoietic$broad_fig1_cell.labels)


In [ ]:
# Adult BM stromal
#FBM_Stromal$celltype <- FBM_Stromal$cell.labels

# Adult BM haematopoietic
FBM_Haematopoietic$celltype <- FBM_Haematopoietic$broad_fig1_cell.labels

#table(FBM_Stromal$celltype)
table(FBM_Haematopoietic$celltype)


<b><font size=5 color=pink >Step 5: load the published 2023 organoid dataset</font></b>


In [ ]:
load(file.path(project_root, "data", "reference", "organoids23", "Organoids23.RData"))
DimPlot(Organoids23, label = TRUE, group.by = "celltype")


In [ ]:
table(Organoids23@meta.data$celltype)


In [ ]:
library(dplyr)

Organoids23$compartment <- dplyr::case_when(
  Organoids23$celltype %in% c(
    "Endothelium",
    "Fibroblast",
    "MSC"
  ) ~ "Stromal",

  Organoids23$celltype %in% c(
    "Erythroid",
    "HSPC",
    "Megakaryocyte",
    "Monocyte",
    "Myeloid Progenitor"
  ) ~ "Haematopoietic",

  TRUE ~ NA_character_
)

table(Organoids23$compartment, useNA = "ifany")

Organoids23_Stromal <- subset(
  Organoids23,
  subset = compartment == "Stromal"
)

Organoids23_Haematopoietic <- subset(
  Organoids23,
  subset = compartment == "Haematopoietic"
)

table(Organoids23_Stromal$celltype)
table(Organoids23_Haematopoietic$celltype)


<b><font size=5 color=pink >Step 6: integrate datasets</font></b>


In [ ]:
library(Seurat)
library(dplyr)

# =========================
# =========================

FBM_Haematopoietic$dataset <- "FBM"
Adult_BM_Haematopoietic$dataset <- "ABM"
Haematopoietic$dataset <- "BMOs"
Organoids23_Haematopoietic$dataset <- "BMO-2023"

DefaultAssay(FBM_Haematopoietic) <- "RNA"
DefaultAssay(Adult_BM_Haematopoietic) <- "RNA"
DefaultAssay(Haematopoietic) <- "RNA"
DefaultAssay(Organoids23_Haematopoietic) <- "RNA"


# =========================
# =========================

Haematopoietic_merged <- merge(
  x = FBM_Haematopoietic,
  y = list(
    Adult_BM_Haematopoietic,
    Haematopoietic,
    Organoids23_Haematopoietic
  ),
  add.cell.ids = c(
    "FBM",
    "ABM",
    "BMOs",
    "BMO-2023"
  ),
  project = "Haematopoietic_Integration"
)

table(Haematopoietic_merged$dataset)

table(Haematopoietic_merged$celltype)


In [ ]:
table(Haematopoietic_merged$celltype, useNA = "ifany")


In [ ]:
Haematopoietic_merged <- NormalizeData(Haematopoietic_merged)
Haematopoietic_merged <- FindVariableFeatures(Haematopoietic_merged, selection.method = "vst", nfeatures = 2000)

all.genes <- rownames(Haematopoietic_merged)
Haematopoietic_merged <- ScaleData(Haematopoietic_merged, features = all.genes)

Haematopoietic_merged <- RunPCA(Haematopoietic_merged, features = VariableFeatures(object = Haematopoietic_merged))


In [ ]:
Haematopoietic_merged <- IntegrateLayers(
  object = Haematopoietic_merged,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  verbose = TRUE
)

Haematopoietic_merged[["RNA"]] <- JoinLayers(Haematopoietic_merged[["RNA"]])


In [ ]:
ElbowPlot(Haematopoietic_merged, ndims = 50)


In [ ]:
Haematopoietic_merged <- FindNeighbors(Haematopoietic_merged, reduction = "integrated.cca", dims = 1:30)
Haematopoietic_merged <- FindClusters(Haematopoietic_merged, resolution = 2)

Haematopoietic_merged <- RunUMAP(Haematopoietic_merged, reduction = "integrated.cca", dims = 1:30)
Haematopoietic_merged <- RunTSNE(Haematopoietic_merged, reduction = "integrated.cca", dims = 1:30)

p1 <- DimPlot(Haematopoietic_merged, reduction = "tsne", label = TRUE) + ggtitle("t-SNE")
p2 <- DimPlot(Haematopoietic_merged, reduction = "umap", label = TRUE) + ggtitle("UMAP")

p1+p2


In [ ]:
DimPlot(Haematopoietic_merged, reduction = "umap",label = TRUE, group.by = "orig.ident")
DimPlot(Haematopoietic_merged, reduction = "umap",label = TRUE, split.by = "orig.ident")


In [ ]:
saveRDS(Haematopoietic_merged, file = file.path(project_root, "data", "processed", "BMOs_Haematopoietic_Final_Integrated_CCA.rds"))


In [ ]:
Haematopoietic_merged <- readRDS(file.path(project_root, "data", "processed", "BMOs_Haematopoietic_Final_Integrated_CCA.rds"))


<b><font size=5 color=pink >Step 7: haematopoietic group-level correlation</font></b>


In [ ]:
table(Haematopoietic_merged@meta.data$dataset)


In [ ]:
table(Haematopoietic_merged@meta.data$group, useNA = "ifany")


In [ ]:
Haematopoietic_merged$group_old <- Haematopoietic_merged$group

Haematopoietic_merged$group <- as.character(Haematopoietic_merged$group)
Haematopoietic_merged$dataset <- as.character(Haematopoietic_merged$dataset)

Haematopoietic_merged$group[is.na(Haematopoietic_merged$group)] <-
  Haematopoietic_merged$dataset[is.na(Haematopoietic_merged$group)]

table(Haematopoietic_merged$group, useNA = "ifany")


In [ ]:
av <- AverageExpression(Haematopoietic_merged,
                        group.by = "group",
                        assays = "RNA")
av <- av[[1]]
head(av)
dim(av)


In [ ]:
cg <- names(tail(sort(apply(av,1,sd)),49455))
#View(av[cg,])
a <- as.matrix(av[cg,])
exp <- cor(x = a,y = NULL, method = "spearman")

pheatmap::pheatmap(exp, display_numbers = TRUE, number_format = "%.2f",
                   fontsize_number = 10,
                   border_color = NA,
                   color = colorRampPalette(c("#2166ac",'#f4f5f4',"#b2182b"))(100))


In [ ]:
cg <- names(tail(sort(apply(av, 1, sd)), 49455))

a <- as.matrix(av[cg, ])

exp <- cor(
  x = a,
  y = NULL,
  method = "spearman"
)

bk <- seq(0.4, 1, length.out = 101)

exp_plot <- exp
exp_plot[exp_plot < 0.4] <- 0.4

num_mat <- matrix(
  sprintf("%.2f", exp),
  nrow = nrow(exp),
  ncol = ncol(exp),
  dimnames = dimnames(exp)
)

pheatmap::pheatmap(
  exp_plot,
  display_numbers = num_mat,
  fontsize_number = 10,
  border_color = NA,
  color = colorRampPalette(c("#2166ac", "#f4f5f4", "#b2182b"))(100),
  breaks = bk,
  legend_breaks = seq(0.4, 1, by = 0.1),
  legend_labels = seq(0.4, 1, by = 0.1)
)


In [ ]:
library(pheatmap)

cg <- names(tail(sort(apply(av, 1, sd)), 49455))

a <- as.matrix(av[cg, ])

exp <- cor(
  x = a,
  y = NULL,
  method = "spearman"
)

pheatmap::pheatmap(
  exp_plot,
  display_numbers = num_mat,
  fontsize_number = 10,
  number_color = "black",
  border_color = NA,
  color = colorRampPalette(c("#2166ac", "#f4f5f4", "#b2182b"))(100),
  breaks = bk,
  legend_breaks = seq(0.4, 1, by = 0.1),
  legend_labels = seq(0.4, 1, by = 0.1),
  filename = file.path(project_root, "results", "figures", "Haematopoietic_group_similarity_heatmap.pdf"),
  width = 5.2,
  height = 5
)


<b><font size=5 color=pink >Step 8: haematopoietic cell-type correlation</font></b>


In [ ]:
library(Seurat)
library(dplyr)
library(pheatmap)

# ============================================================
# ============================================================

DefaultAssay(Haematopoietic_merged) <- "RNA"

Haematopoietic_merged$celltype <- as.character(Haematopoietic_merged$celltype)
Haematopoietic_merged$dataset <- as.character(Haematopoietic_merged$dataset)

Haematopoietic_merged$group_plot <- as.character(Haematopoietic_merged$group)

idx_na <- is.na(Haematopoietic_merged$group_plot) |
  Haematopoietic_merged$group_plot == ""

Haematopoietic_merged$group_plot[idx_na] <-
  Haematopoietic_merged$dataset[idx_na]

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("Adult_BM", "AdultBM", "Adult BM")
] <- "ABM"

Haematopoietic_merged$group_plot[
  Haematopoietic_merged$group_plot %in% c("Organoids23", "BMO2023", "BMO_2023")
] <- "BMO-2023"

table(Haematopoietic_merged$group_plot, useNA = "ifany")
table(Haematopoietic_merged$celltype, useNA = "ifany")


In [ ]:
# ============================================================
# ============================================================

keep_groups <- c(
  "ABM",
  "FBM",
  "Static.25d",
  "Dynamic.25d",
  "Dynamic.31d"
)

Haematopoietic_sim <- subset(
  Haematopoietic_merged,
  subset = group_plot %in% keep_groups &
    !is.na(celltype) &
    celltype != ""
)

Haematopoietic_sim$group_celltype <- paste(
  Haematopoietic_sim$group_plot,
  Haematopoietic_sim$celltype,
  sep = " | "
)

table(Haematopoietic_sim$group_plot, useNA = "ifany")
table(Haematopoietic_sim$group_celltype)


In [ ]:
library(dplyr)
library(tidyr)

keep_groups <- c(
  "ABM",
  "FBM",
  "Static.25d",
  "Dynamic.25d",
  "Dynamic.31d"
)

cell_count_df <- Haematopoietic_merged@meta.data %>%
  dplyr::mutate(
    group_plot = as.character(group_plot),
    celltype = as.character(celltype)
  ) %>%
  dplyr::filter(
    group_plot %in% keep_groups,
    !is.na(celltype),
    celltype != ""
  ) %>%
  dplyr::count(group_plot, celltype, name = "n_cells") %>%
  dplyr::arrange(group_plot, celltype)


In [ ]:
cell_count_wide <- cell_count_df %>%
  tidyr::pivot_wider(
    names_from = group_plot,
    values_from = n_cells,
    values_fill = 0
  )

cell_count_wide


In [ ]:
min_cells <- 30

keep_group_celltype <- cell_count_df %>%
  dplyr::filter(n_cells >= min_cells) %>%
  dplyr::mutate(group_celltype = paste(group_plot, celltype, sep = " | ")) %>%
  dplyr::pull(group_celltype)

Haematopoietic_sim <- Haematopoietic_merged

Haematopoietic_sim$group_celltype <- paste(
  Haematopoietic_sim$group_plot,
  Haematopoietic_sim$celltype,
  sep = " | "
)

Haematopoietic_sim <- subset(
  Haematopoietic_sim,
  subset = group_plot %in% keep_groups &
    group_celltype %in% keep_group_celltype
)

table(Haematopoietic_sim$group_celltype)


In [ ]:
# ============================================================
# ============================================================

av <- AverageExpression(
  Haematopoietic_sim,
  group.by = "group_celltype",
  assays = "RNA",
  slot = "data"
)

av <- av[[1]]
av <- as.matrix(av)

dim(av)
head(av[, 1:min(5, ncol(av))])


In [ ]:
# ============================================================
# ============================================================

av <- av[
  rowSums(!is.finite(av)) == 0,
  ,
  drop = FALSE
]

av <- av[
  rowSums(av) > 0,
  ,
  drop = FALSE
]

gene_sd <- apply(av, 1, sd)

av_use <- av[
  !is.na(gene_sd) & gene_sd > 0,
  ,
  drop = FALSE
]

dim(av_use)

exp <- cor(
  x = av_use,
  y = NULL,
  method = "spearman",
  use = "pairwise.complete.obs"
)

dim(exp)


In [ ]:
# ============================================================
# ============================================================

outdir <- file.path(project_root, "results", "figures", "reference_integration")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

pheatmap::pheatmap(
  exp,
  display_numbers = FALSE,
  number_format = "%.2f",
  fontsize_number = 6,
  fontsize_row = 6,
  fontsize_col = 6,
  color = colorRampPalette(c("#2166ac", "#f4f5f4", "#b2182b"))(100),
  filename = file.path(outdir, "Haematopoietic_group_celltype_similarity_heatmap.pdf"),
  width = 14,
  height = 12
)


In [ ]:
# ============================================================
# ============================================================

outdir <- file.path(project_root, "results", "figures", "reference_integration")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

bk <- seq(0.4, 1, length.out = 101)

exp_plot <- exp
exp_plot[exp_plot < 0.4] <- 0.4

pheatmap::pheatmap(
  exp_plot,
  display_numbers = FALSE,
  number_format = "%.2f",
  fontsize_number = 6,
  fontsize_row = 6,
  fontsize_col = 6,
  color = colorRampPalette(c("#2166ac", "#f4f5f4", "#b2182b"))(100),
  breaks = bk,
  legend_breaks = seq(0.4, 1, by = 0.1),
  legend_labels = seq(0.4, 1, by = 0.1),
  border_color = NA,
  filename = file.path(outdir, "Haematopoietic_group_celltype_similarity_heatmap.pdf"),
  width = 8,
  height = 7
)
